#  **멀티모달(multi-modal) 학습** 

- **멀티모달 학습**은 텍스트, 이미지, 오디오 등 **다양한 데이터 형식**을 동시에 처리하는 혁신적인 AI 접근 방식

    - 텍스트: 자연어 문장, 문서, 메타데이터
    - 이미지: 사진, 그래프, 차트, 다이어그램
    - 오디오: 음성, 음악, 환경음
    - 비디오: 동영상 시퀀스
    - 센서 데이터: IoT 센서, 생체신호 등

- 인간의 감각 통합 방식과 유사하게 여러 정보 채널을 **동시에 학습**하여 더 깊은 맥락적 이해 제공

- **모달리티별 특화된 신경망** 아키텍처를 사용하여 각 데이터 유형의 고유한 특징을 효과적으로 추출

    - **텍스트 모달리티**: **BERT, GPT** 등의 언어 모델을 활용하여 문맥적 이해와 의미 표현 학습
    - **이미지 모달리티**: **CNN, Vision Transformer**를 통해 시각적 특징 및 패턴 추출
    - **오디오 모달리티**: **웨이브넷, Mel-스펙트로그램** 기반 모델로 음향학적 특성 분석


- 멀티 모달리티 **특징 융합 방법**

    - **Early Fusion**: 입력 단계에서 모달리티 특징을 **직접 결합**하는 방식

    - **Late Fusion**: 각 모달리티를 **개별적으로 처리** 후 최종 결과 통합

    - **Hybrid Fusion**: **다단계 특징 결합**을 통한 유연하고 복합적인 모달리티 통합 접근법

---

# **학습 목표**

이 노트북을 통해 다음을 학습합니다:

- **CLIP 모델을 활용한 멀티모달 임베딩 이해**: 이미지와 텍스트를 공통 벡터 공간에 투영하는 원리 학습
- **LangChain에서 이미지-텍스트 유사도 검색 구현**: OpenCLIPEmbeddings를 활용한 멀티모달 검색 시스템 구현
- **FAISS 벡터 스토어와 멀티모달 임베딩 통합**: 대규모 이미지 데이터셋에서 효율적인 유사도 검색 구현

---

## **사전 준비**

### 필수 환경
- Python 3.10 이상

### 필수 패키지
다음 패키지들이 설치되어 있어야 합니다:
```bash
pip install --upgrade --quiet langchain-experimental open-clip-torch faiss-cpu pillow
```

### 테스트 이미지 파일
- COCO 데이터셋 샘플 이미지 (노트북 내에서 자동 다운로드)
- Unsplash 25K 이미지 데이터셋 (노트북 내에서 자동 다운로드)

---

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

---

## **이미지-텍스트 임베딩 기술**

- **이미지-텍스트 임베딩**의 핵심은 서로 다른 모달리티 데이터를 **공통 벡터 공간에 투영**하는 기술

- 모달리티 간 **의미적 유사성을 벡터로 표현**하여 cross-modal 이해와 분석 가능

- **CLIP 모델**은 이미지와 텍스트를 **의미론적으로 연결**하는 혁신적인 접근 방식 제시

    - OpenAI의 **12명의 연구진**이 개발했으며, 이미지-텍스트 쌍으로 사전 학습
    
    - GPT 모델과 유사하게 **제로샷 학습** 능력을 보유하여 새로운 작업에 빠르게 적용 가능

    - 특정 작업에 대한 **추가 학습 없이** 자연어 기반의 이미지 분석이 가능

    - 논문: https://arxiv.org/abs/2103.00020

### **OpenClip** 소개

- OpenClip은 OpenAI의 CLIP(Contrastive Language-Image Pre-training)을 오픈소스로 구현한 라이브러리

    - 이미지와 텍스트 간의 의미적 유사성 계산
    - 다양한 크기와 성능의 모델 지원
    - 이미지 검색, 분류, 설명 생성 등 다양한 응용 가능

### 1. **필요 라이브러리 임포트** 

In [ ]:
# 이미지 처리를 위한 라이브러리 임포트
import numpy as np
import os
import requests
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
import torch
import open_clip
from langchain_experimental.open_clip import OpenCLIPEmbeddings

# 파이토치 버전 확인
print(torch.__version__)

### 2. **CLIP 임베딩 모델 로드** 


In [ ]:
# 사용 가능한 모델 확인
def check_available_models():
    models = open_clip.list_pretrained()
    print("사용 가능한 모델 목록:")
    for model in models:
        print(f"- {model}")


check_available_models()

In [ ]:
# 사전 학습된 CLIP 모델 로드
# 참고: langchain-experimental의 OpenCLIPEmbeddings는 실험적 기능입니다.
# 대안: https://github.com/mlfoundations/open_clip

clip_embd_model = OpenCLIPEmbeddings(model_name="ViT-B-16-quickgelu", checkpoint="openai")

clip_embd_model

### 3. **이미지 및 텍스트 임베딩 생성** 

In [ ]:
import urllib

# 샘플 이미지 다운로드
urls = [
    "http://images.cocodataset.org/val2017/000000000285.jpg",  # bear
    "http://images.cocodataset.org/val2017/000000039769.jpg",  # cats
    "http://images.cocodataset.org/val2017/000000000776.jpg",  # Teddy bears

]

# 이미지와 비교할 텍스트 쿼리 또는 이미지에 대한 설명 텍스트
texts = ["bear", "cats", "Teddy bears"]  

for url, text in zip(urls, texts):
    fname = text.replace(" ", "_").lower()
    urllib.request.urlretrieve(url, f"{fname}.jpg")


In [ ]:
# 이미지 출력
def show_image(image_path):
    image = Image.open(image_path)
    plt.imshow(image)
    plt.axis('off')
    plt.show()

# 이미지 경로
image_paths = [f"{text.replace(' ', '_').lower()}.jpg" for text in texts]

# 이미지 출력
for image_path in image_paths:
    show_image(image_path)

In [ ]:
# 이미지와 텍스트 임베딩 생성
img_features = clip_embd_model.embed_image(image_paths)
text_features = clip_embd_model.embed_documents(texts)

# 임베딩을 NumPy 배열로 변환
img_features_np = np.array(img_features)
text_features_np = np.array(text_features)

# 배열의 모양 확인
print("Image features shape:", img_features_np.shape)
print("Text features shape:", text_features_np.shape)

In [ ]:
text_features

In [ ]:
# 이미지-텍스트 유사도 계산 및 시각화
similarity = np.matmul(text_features_np, img_features_np.T)

# 결과 시각화
count = len(texts)
plt.figure(figsize=(20, 14))
plt.imshow(similarity, vmin=0.1, vmax=0.3)
plt.yticks(range(count), texts, fontsize=18)
plt.xticks([])

# 이미지 표시
for i, image in enumerate([Image.open(image_path) for image_path in image_paths]):
    plt.imshow(image, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")

# 유사도 점수 표시
for x in range(similarity.shape[1]):
    for y in range(similarity.shape[0]):
        plt.text(x, y, f"{similarity[y, x]:.2f}", ha="center", va="center", size=12)

# 그래프 테두리 제거
for side in ["left", "top", "right", "bottom"]:
    plt.gca().spines[side].set_visible(False)

plt.xlim([-0.5, count - 0.5])
plt.ylim([count + 0.5, -2])
plt.title("Cosine Similiarity", size=20)

---

## **UNSPLASH 데이터셋을 활용한 이미지 검색**

- **Faiss** 라이브러리를 활용하여 **대규모 이미지 검색**을 구현

- Faiss **인덱스**를 파일로 저장하고 필요할 때 로드하는 방식으로 메모리를 관리

- **유사도 검색** 알고리즘으로 대규모 데이터셋에서도 빠른 검색이 가능

### 1. **필요 라이브러리 설치** 

- faiss-cpu 설치 (GPU 사용 시 faiss-gpu 설치)
- pip install faiss-cpu sentence-transformers 

In [ ]:
import os
import zipfile
from tqdm.autonotebook import tqdm
from PIL import Image
import requests
import torch
import numpy as np
import faiss 
import pickle
from IPython.display import display
from sentence_transformers import util

### 2. **UNSPLASH 이미지 데이터셋 준비** 

In [ ]:
# 이미지 다운로드 
img_folder = 'unsplash/'
if not os.path.exists(img_folder) or len(os.listdir(img_folder)) == 0:
    os.makedirs(img_folder, exist_ok=True)
    photo_filename = 'unsplash-25k-photos.zip'
    if not os.path.exists(photo_filename):
        util.http_get('http://sbert.net/datasets/'+photo_filename, photo_filename) # sentence-transformers util 사용
    with zipfile.ZipFile(photo_filename, 'r') as zf:
        for member in tqdm(zf.infolist(), desc='Extracting'):
            zf.extract(member, img_folder)

# 이미지 파일 경로 리스트 생성
img_files = glob(os.path.join(img_folder, '*.jpg'))

# 이미지 파일 경로 리스트 확인
print(f'이미지 파일 개수: {len(img_files)}')
print(img_files[:5])

### 3. **FAISS 인덱스에 임베딩 저장** 

In [ ]:
# 이미지 파일 경로 리스트 생성
img_files = glob(os.path.join(img_folder, '*.jpg'))

# 이미지 파일 경로 리스트 확인
print(f'이미지 파일 개수: {len(img_files)}')
print(img_files[:5])

In [ ]:
# 임베딩 모델 로드
clip_embd_model = OpenCLIPEmbeddings(model_name="ViT-B-16-quickgelu", checkpoint="openai")

# 이미지 폴더에서 이미지를 불러와서 임베딩을 계산하고 FIASS에 저장 
# 테스트를 위해 1000개의 이미지만 사용하는 것으로 수정 (전체 이미지 사용 시 시간이 오래 걸릴 수 있음)

img_files = img_files[:1000]

# 이미지 임베딩 계산 - batch 처리
img_embeddings = []
batch_size = 100
for i in tqdm(range(0, len(img_files), batch_size), desc='Calculating Image Embeddings'):
    batch_files = img_files[i:i+batch_size]
    batch_embeddings = clip_embd_model.embed_image(batch_files)
    img_embeddings.append(batch_embeddings)

# 이미지 임베딩을 NumPy 배열로 변환
img_embeddings = np.vstack(img_embeddings)

# 이미지 임베딩의 크기 확인
print(f'Image Embeddings Shape: {img_embeddings.shape}')

In [ ]:
# FAISS 인덱스 생성
index = faiss.IndexFlatL2(img_embeddings.shape[1])
index.add(img_embeddings)  # 이미지 임베딩 추가

# FAISS 인덱스 저장
# 권장: 인덱스 파일명은 모델명이나 데이터셋 정보를 포함하여 고유성을 확보하세요.
# 예: 'clip_vit_b16_unsplash_1k.index', 'openai_clip_coco_embeddings.index'
faiss.write_index(index, 'img_embeddings.index')

In [ ]:
# FAISS 인덱스 로드
index = faiss.read_index('img_embeddings.index')

# FAISS 인덱스 확인
print(f"FAISS index: {index}")

In [ ]:
# FAISS 인덱스에서 가장 가까운 k개의 이웃 검색
def search_similar_images(query_image_path, k):
    # 이미지 임베딩 생성
    query_embedding = clip_embd_model.embed_image([query_image_path])
    
    # 리스트를 numpy 배열로 변환
    query_embedding = np.array(query_embedding, dtype=np.float32)
    
    # FAISS 인덱스에서 가장 가까운 k개의 이웃 검색
    distances, indices = index.search(query_embedding, k)
    
    # 결과 반환
    return distances[0], indices[0]


# 쿼리 이미지 경로
query_image_path = img_files[0]  # 첫 번째 이미지 사용

# 쿼리 이미지 출력
show_image(query_image_path)

In [ ]:
# 유사한 이미지 검색
distances, indices = search_similar_images(query_image_path, k=5)

# 유사한 이미지 출력
for i, idx in enumerate(indices):
    print(f"유사한 이미지 {i+1}: {img_files[idx]} (거리: {distances[i]:.4f})")
    show_image(img_files[idx])

In [ ]:
# 텍스트 쿼리 임베딩 생성
text_queries = [
    "a black dog running in the park",
    "공원에서 달리는 검은 개",
]

# FAISS 인덱스에서 가장 가까운 k개의 이웃 검색
def search_similar_images_by_text(text_query, k):
    # 텍스트 쿼리 임베딩 생성
    text_embedding = clip_embd_model.embed_documents([text_query])
    
    # 리스트를 numpy 배열로 변환
    text_embedding = np.array(text_embedding, dtype=np.float32)
    
    # FAISS 인덱스에서 가장 가까운 k개의 이웃 검색
    distances, indices = index.search(text_embedding, k)
    
    # 결과 반환
    return distances[0], indices[0]


# 텍스트 쿼리 이미지 검색
for text_query in text_queries:
    print(f"텍스트 쿼리: {text_query}")
    
    # 유사한 이미지 검색
    distances, indices = search_similar_images_by_text(text_query, k=5)
    
    # 유사한 이미지 출력
    for i, idx in enumerate(indices):
        print(f"유사한 이미지 {i+1}: {img_files[idx]} (거리: {distances[i]:.4f})")
        show_image(img_files[idx])

---
## [실습] 

- 다른 텍스트를 이용해서 이미지를 검색하고, 그 결과를 분석하세요. 



In [ ]:
# 다양한 텍스트 쿼리로 이미지 검색 테스트
text_queries = [
    "sunset over mountains",
    "산 위의 일몰",
    "people walking on the street",
    "거리를 걷는 사람들",
    "a beautiful flower",
    "아름다운 꽃",
]


# 여기에 코드를 작성하세요. 

---
## [심화] 한국어 지원 모델

- 다국어 지원하는 CLIP 모델을 사용하여, FAISS 인덱스를 생성합니다. (예: xlm-roberta-base-ViT-B-32)
    - **XLM-RoBERTa**는 Facebook AI에서 개발한 다국어 언어 모델로, 100개 언어에 대해 사전 학습
    - **ViT-B/32**는 이미지 처리용 Vision Transformer로, B는 모델 크기(Base), 32는 패치 크기를 의미
    - **OpenCLIP**에서는 XLM-RoBERTa를 텍스트 인코더로, ViT를 이미지 인코더로 조합하여 사용

- 영어, 한국어 쿼리에 대한 이미지 검색을 수행합니다. 

In [ ]:
# 여기에 코드를 작성하세요.
